## Projet IA — Notebook 2 : Démo Agent LLM
## Analyseur Intelligent de Scènes de Conduite — Module B

Scénario : Conduite de Nuit | Framework : Groq API

Ce notebook démontre l'agent IA complet :
1. Connexion à l'API Groq (gratuit)
2. Les 3 outils de l'agent (météo, distance, règles de conduite)
3. La boucle agent : observation → réflexion → action → réponse
4. Tests sur des scènes variées avec visualisation des résultats


## Installation & Configuration

In [ ]:
!pip install groq requests -q
print("✅ Installation terminée")


In [ ]:
import os, json, requests
from groq import Groq

# ─────────────────────────────────────────────────────────

GROQ_API_KEY = "gsk_aFIQRSfGzXhNFuWDVm10WGdyb3FtrerdtyghubytgfdezqéaqzertYBkHJX6mIJFgwESUDiZvdjSk8"   # Notre clé Groq

os.environ['GROQ_API_KEY']         = GROQ_API_KEY
os.environ['OPENWEATHER_API_KEY']  = OWM_API_KEY

client = Groq(api_key=GROQ_API_KEY)
MODELE = "llama-3.3-70b-versatile"

print(f"✅ Client Groq configuré — modèle : {MODELE}")

# Test rapide de connexion
try:
    test = client.chat.completions.create(
        model=MODELE,
        messages=[{"role": "user", "content": "Dis juste 'OK'"}],
        max_tokens=5
    )
    print(f"✅ Connexion API OK : {test.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ Erreur connexion : {e}")
    print("   → Vérifiez votre clé GROQ_API_KEY")


## Définition des Outils de l'Agent

In [ ]:
# ─── LES 3 OUTILS DE L'AGENT ─────────────────────────────────────────────────
# Format JSON standard

OUTILS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retourne la météo actuelle d'une ville. Utilisé pour évaluer la visibilité.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nom de la ville, ex: 'Paris'"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "estimate_distance",
            "description": "Estime la distance d'un objet en mètres depuis la taille de sa bounding box.",
            "parameters": {
                "type": "object",
                "properties": {
                    "bbox_height_ratio": {
                        "type": "number",
                        "description": "Hauteur de la bbox / hauteur de l'image (entre 0 et 1)"
                    },
                    "object_type": {
                        "type": "string",
                        "description": "Type d'objet : 'pedestrian', 'car', 'truck' ou 'bus'"
                    }
                },
                "required": ["bbox_height_ratio", "object_type"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_night_driving_rules",
            "description": "Retourne les règles du code de la route pour la conduite de nuit.",
            "parameters": {
                "type": "object",
                "properties": {
                    "country": {"type": "string", "description": "Pays, ex: 'France'"}
                },
                "required": ["country"]
            }
        }
    },
]

print("✅ 3 outils définis pour l'agent :")
for outil in OUTILS:
    print(f"   🔧 {outil['function']['name']} — {outil['function']['description'][:60]}...")


In [ ]:
# ─── IMPLÉMENTATION DES OUTILS ────────────────────────────────────────────────

def get_weather(city):
    """Météo réelle via OpenWeatherMap, ou simulée si pas de clé API."""
    owm_key = os.environ.get('OPENWEATHER_API_KEY', '')
    if not owm_key:
        # Données simulées — suffisantes pour la démo
        return {
            "city": city, "description": "nuit claire", "temperature": 11,
            "humidity": 72, "wind_speed": 3.5, "visibility_m": 8500,
            "note": "Données simulées (ajoutez une clé OWM pour données réelles)"
        }
    try:
        url  = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={owm_key}&units=metric&lang=fr"
        data = requests.get(url, timeout=5).json()
        return {
            "city": city,
            "description": data["weather"][0]["description"],
            "temperature": data["main"]["temp"],
            "humidity":    data["main"]["humidity"],
            "wind_speed":  data["wind"]["speed"],
            "visibility_m": data.get("visibility", 10000),
        }
    except Exception as e:
        return {"city": city, "erreur": str(e), "visibility_m": 9000}


def estimate_distance(bbox_height_ratio, object_type):
    """
    Estimation de distance par similitude de triangles.
    Principe : distance = hauteur_réelle_objet / taille_apparente × constante
    La nuit on ajoute +20% d'incertitude (contraste réduit = bbox moins précise).
    """
    # Hauteur réelle approximative en mètres
    hauteurs = {'pedestrian': 1.75, 'car': 1.50, 'truck': 3.50, 'bus': 3.20}

    if bbox_height_ratio <= 0.01:
        return {"distance_m": None, "proximite": "indéterminée"}

    h_reel    = hauteurs.get(object_type, 1.75)
    distance  = (h_reel / bbox_height_ratio) * 0.9   # facteur optique
    dist_nuit = round(distance * 1.2, 1)              # +20% incertitude nocturne

    if   dist_nuit < 8:  proximite = "DANGER IMMÉDIAT (< 8m)"
    elif dist_nuit < 20: proximite = "Proche — vigilance maximale"
    elif dist_nuit < 40: proximite = "Distance moyenne"
    else:                proximite = "Éloigné"

    return {
        "distance_estimee_m": dist_nuit,
        "object_type": object_type,
        "proximite": proximite,
        "note": "Estimation ±20% (conditions nocturnes)"
    }


def get_night_driving_rules(country="France"):
    """Base de règles de conduite nocturne (code de la route)."""
    regles = {
        "France": {
            "vitesse_hors_agglomeration_kmh": 80,
            "vitesse_agglomeration_kmh": 50,
            "vitesse_mauvaise_visibilite_kmh": 50,
            "distance_securite_min_m": 15,
            "feux": "Feux de croisement obligatoires dès la nuit tombée",
            "alertes_prioritaires": [
                "Piéton sans équipement réfléchissant",
                "Visibilité < 50m → arrêt obligatoire",
                "Éblouissement — ne pas fixer les phares adverses",
                "Somnolence — s'arrêter toutes les 2h"
            ],
            "conseils": [
                "Réduire la vitesse de 20% par rapport aux limites de jour",
                "Maintenir minimum 15m de distance avec le véhicule devant",
                "Surveiller les bords de route (piétons, animaux)",
                "Vérifier l'état des phares avant tout trajet nocturne"
            ]
        }
    }
    return regles.get(country, regles["France"])


def executer_outil(nom, arguments):
    """Dispatch : appelle le bon outil selon la demande de l'agent."""
    if   nom == "get_weather":           result = get_weather(**arguments)
    elif nom == "estimate_distance":     result = estimate_distance(**arguments)
    elif nom == "get_night_driving_rules": result = get_night_driving_rules(**arguments)
    else:                                result = {"erreur": f"Outil inconnu : {nom}"}
    return json.dumps(result, ensure_ascii=False, indent=2)


# Test rapide des outils
print("🧪 Test des outils :")
print("\n📍 Météo Paris :")
print(json.dumps(get_weather("Paris"), indent=2, ensure_ascii=False))
print("\n📏 Distance piéton (bbox_height=0.35) :")
print(json.dumps(estimate_distance(0.35, "pedestrian"), indent=2, ensure_ascii=False))


## 🧠 Étape 2 — Le System Prompt de l'Agent

In [ ]:
# ─── SYSTEM PROMPT ────────────────────────────────────────────────────────────
# C'est le "cerveau" de l'agent : il définit son rôle, ses responsabilités
# et le format exact de réponse attendu.

SYSTEM_PROMPT = """Tu es un expert en sécurité routière spécialisé dans la conduite de nuit.

Tu reçois des résultats de détection YOLO (objets détectés dans une image dashcam nocturne).
Ton rôle : analyser la scène et produire un rapport de sécurité structuré en français.

RÈGLES ABSOLUES :
1. Toujours appeler get_night_driving_rules('France') pour connaître les règles applicables
2. Appeler estimate_distance pour chaque piéton et véhicule proche détecté
3. Appeler get_weather pour évaluer la visibilité météo
4. Utiliser ces informations pour calibrer le niveau de risque

NIVEAUX DE RISQUE :
- FAIBLE   : route dégagée, conditions normales, aucun piéton proche
- MOYEN    : quelques véhicules, piétons à distance, conditions acceptables
- ÉLEVÉ    : piéton proche OU densité élevée d'objets OU mauvaise visibilité
- CRITIQUE : piéton < 8m OU plusieurs dangers combinés OU visibilité < 50m

FORMAT DE RÉPONSE OBLIGATOIRE (respecter exactement) :
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANALYSE DE SCÈNE — CONDUITE DE NUIT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CONTEXTE : [description de la scène en 1 phrase]

OBJETS DÉTECTÉS :
  • [classe] : [nombre] objet(s) — confiance moy. [X%]

DISTANCES ESTIMÉES :
  • [objet] : ~[X]m → [niveau de proximité]

NIVEAU DE RISQUE : [FAIBLE / MOYEN / ÉLEVÉ / CRITIQUE]

ANALYSE : [explication du risque en 2-3 phrases]

RECOMMANDATIONS :
  • [conseil concret 1]
  • [conseil concret 2]
  • [conseil concret 3]

CONDITIONS MÉTÉO : [ville] — [description météo] — Visibilité : [X]m
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━"""

print("✅ System prompt défini")
print(f"   Longueur : {len(SYSTEM_PROMPT)} caractères")
print(f"   Outils requis : get_night_driving_rules, estimate_distance, get_weather")


## La Boucle Agent (Observation → Action → Réponse)

In [ ]:
# ─── BOUCLE PRINCIPALE DE L'AGENT ────────────────────────────────────────────
# Principe : l'agent peut appeler plusieurs outils avant de répondre
# Boucle : user → agent (outil?) → outil → agent (outil?) → ... → réponse finale

def run_agent(detections, city="Paris", verbose=True):
    """
    Lance l'agent sur une liste de détections YOLO.

    L'agent va :
    1. Recevoir les détections (JSON)
    2. Décider quels outils appeler (météo, distance, règles)
    3. Appeler les outils et recevoir leurs résultats
    4. Synthétiser tout en un rapport structuré

    Returns : rapport texte complet
    """
    # Message utilisateur avec les détections YOLO
    message_user = f"""Analyse cette scène de conduite nocturne.

Détections YOLO (format JSON) :
{json.dumps(detections, ensure_ascii=False, indent=2)}

Ville : {city}
Génère un rapport de sécurité complet."""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": message_user},
    ]

    outils_utilises = []

    # Boucle agent : max 8 tours pour éviter les boucles infinies
    for tour in range(8):
        if verbose:
            print(f"   Tour {tour+1}...", end=" ")

        reponse = client.chat.completions.create(
            model       = MODELE,
            messages    = messages,
            tools       = OUTILS,
            tool_choice = "auto",     # l'agent choisit lui-même
            max_tokens  = 2000,
            temperature = 0.3,        # déterministe pour la sécurité
        )

        msg = reponse.choices[0].message

        # Si pas d'appel d'outil → réponse finale
        if not msg.tool_calls:
            if verbose:
                print(f"→ Réponse finale ({reponse.usage.total_tokens} tokens)")
            return msg.content, outils_utilises

        # L'agent veut appeler des outils
        noms_outils = [tc.function.name for tc in msg.tool_calls]
        outils_utilises.extend(noms_outils)
        if verbose:
            print(f"→ Appel outil(s) : {noms_outils}")

        # On ajoute la réponse de l'agent à l'historique
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id, "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in msg.tool_calls
            ],
        })

        # On exécute chaque outil et on renvoie le résultat à l'agent
        for tc in msg.tool_calls:
            args    = json.loads(tc.function.arguments)
            resultat = executer_outil(tc.function.name, args)
            messages.append({
                "role":         "tool",
                "tool_call_id": tc.id,
                "content":      resultat,
            })

    return "L'agent n'a pas pu conclure après 8 tours.", outils_utilises


print("✅ Boucle agent définie")
print("   Fonctionnement : observation → choix d'outil → exécution → synthèse")


## Étape 4 — Tests sur des Scènes Variées

In [ ]:
# ─── SCÈNE 1 : Route tranquille (RISQUE FAIBLE attendu) ───────────────────────

scene_1 = [
    {"class_name": "car", "confidence": 0.91,
     "bbox": {"x1": 0.10, "y1": 0.55, "x2": 0.35, "y2": 0.75},
     "bbox_height": 0.20},
    {"class_name": "traffic light", "confidence": 0.88,
     "bbox": {"x1": 0.48, "y1": 0.20, "x2": 0.54, "y2": 0.38},
     "bbox_height": 0.18},
]

print("=" * 60)
print("TEST 1 : Route avec une voiture + feu de signalisation")
print("=" * 60)
print("Détections :")
for d in scene_1:
    print(f"   {d['class_name']:15s} conf={d['confidence']:.0%}")
print()
print("L'agent analyse la scène...")
rapport_1, outils_1 = run_agent(scene_1, city="Lyon", verbose=True)
print(f"   Outils utilisés : {outils_1}")
print()
print("RAPPORT :")
print(rapport_1)


In [ ]:
# ─── SCÈNE 2 : Piéton très proche (RISQUE CRITIQUE attendu) ──────────────────

scene_2 = [
    {"class_name": "pedestrian", "confidence": 0.72,
     "bbox": {"x1": 0.68, "y1": 0.30, "x2": 0.78, "y2": 0.90},
     "bbox_height": 0.60},     # ← grande bbox = objet très proche !
    {"class_name": "car", "confidence": 0.85,
     "bbox": {"x1": 0.05, "y1": 0.55, "x2": 0.40, "y2": 0.80},
     "bbox_height": 0.25},
    {"class_name": "truck", "confidence": 0.78,
     "bbox": {"x1": 0.60, "y1": 0.40, "x2": 0.90, "y2": 0.85},
     "bbox_height": 0.45},
]

print("=" * 60)
print("TEST 2 : Piéton très proche + camion (danger critique)")
print("=" * 60)
print("Détections :")
for d in scene_2:
    print(f"   {d['class_name']:15s} conf={d['confidence']:.0%}  bbox_height={d['bbox_height']:.2f}")
print()
print("L'agent analyse la scène...")
rapport_2, outils_2 = run_agent(scene_2, city="Paris", verbose=True)
print(f"   Outils utilisés : {outils_2}")
print()
print("RAPPORT :")
print(rapport_2)


In [ ]:
# ─── SCÈNE 3 : Intersection chargée (RISQUE ÉLEVÉ attendu) ───────────────────

scene_3 = [
    {"class_name": "pedestrian",   "confidence": 0.68,
     "bbox": {"x1": 0.20, "y1": 0.50, "x2": 0.26, "y2": 0.75}, "bbox_height": 0.25},
    {"class_name": "pedestrian",   "confidence": 0.61,
     "bbox": {"x1": 0.75, "y1": 0.52, "x2": 0.80, "y2": 0.74}, "bbox_height": 0.22},
    {"class_name": "car",          "confidence": 0.93,
     "bbox": {"x1": 0.00, "y1": 0.55, "x2": 0.25, "y2": 0.78}, "bbox_height": 0.23},
    {"class_name": "bus",          "confidence": 0.87,
     "bbox": {"x1": 0.65, "y1": 0.35, "x2": 0.95, "y2": 0.90}, "bbox_height": 0.55},
    {"class_name": "traffic sign", "confidence": 0.79,
     "bbox": {"x1": 0.10, "y1": 0.15, "x2": 0.18, "y2": 0.35}, "bbox_height": 0.20},
    {"class_name": "traffic light","confidence": 0.82,
     "bbox": {"x1": 0.50, "y1": 0.10, "x2": 0.57, "y2": 0.30}, "bbox_height": 0.20},
]

print("=" * 60)
print("TEST 3 : Intersection urbaine chargée")
print("=" * 60)
print("Détections :")
for d in scene_3:
    print(f"   {d['class_name']:15s} conf={d['confidence']:.0%}")
print()
print("L'agent analyse la scène...")
rapport_3, outils_3 = run_agent(scene_3, city="Marseille", verbose=True)
print(f"   Outils utilisés : {outils_3}")
print()
print("RAPPORT :")
print(rapport_3)


##Visualisation Comparative des Résultats

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

# ─── Extraction automatique du niveau de risque depuis le rapport ─────────────
def extraire_risque(rapport):
    for niveau in ['CRITIQUE', 'ÉLEVÉ', 'MOYEN', 'FAIBLE']:
        if niveau in rapport.upper().replace('É', 'É'):
            return niveau
    return 'INCONNU'

niveaux = [extraire_risque(r) for r in [rapport_1, rapport_2, rapport_3]]
scenes  = ['Scène 1\n(route calme)', 'Scène 2\n(piéton proche)', 'Scène 3\n(intersection)']
couleurs_risque = {
    'FAIBLE'  : '#27ae60',
    'MOYEN'   : '#f39c12',
    'ÉLEVÉ'   : '#e67e22',
    'CRITIQUE': '#e74c3c',
    'INCONNU' : '#95a5a6',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Résultats Agent LLM — 3 Scènes Nocturnes", fontsize=14, fontweight='bold')

# ── Niveaux de risque ─────────────────────────────────────────────────────────
couleurs = [couleurs_risque.get(n, '#95a5a6') for n in niveaux]
barres   = axes[0].bar(scenes, [1, 1, 1], color=couleurs, alpha=0.85, edgecolor='white', linewidth=2)
for bar, niveau in zip(barres, niveaux):
    axes[0].text(bar.get_x() + bar.get_width()/2, 0.5, niveau,
                 ha='center', va='center', fontsize=13, fontweight='bold', color='white')
axes[0].set_ylim(0, 1.3)
axes[0].set_yticks([])
axes[0].set_title("Niveau de risque par scène", fontweight='bold')

legende = [mpatches.Patch(color=c, label=n) for n, c in couleurs_risque.items() if n != 'INCONNU']
axes[0].legend(handles=legende, loc='upper right', fontsize=9)

# ── Nombre d'objets et d'outils utilisés ─────────────────────────────────────
nb_objets = [len(scene_1), len(scene_2), len(scene_3)]
nb_outils = [len(set(outils_1)), len(set(outils_2)), len(set(outils_3))]

x     = range(len(scenes))
width = 0.35
axes[1].bar([xi - width/2 for xi in x], nb_objets, width, label='Objets détectés', color='#3498db', alpha=0.85)
axes[1].bar([xi + width/2 for xi in x], nb_outils, width, label='Outils appelés',  color='#e74c3c', alpha=0.85)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(scenes)
axes[1].set_ylabel("Nombre")
axes[1].set_title("Objets détectés vs Outils utilisés", fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/resultats_agent.png', dpi=120, bbox_inches='tight')
plt.show()
print("📊 Graphique des résultats sauvegardé")


In [ ]:
# ─── TABLEAU RÉCAPITULATIF ────────────────────────────────────────────────────

print("\n" + "="*70)
print("📊 RÉCAPITULATIF — AGENT LLM (Conduite de Nuit)")
print("="*70)
print(f"{'Scène':25s} {'Objets':8s} {'Outils':8s} {'Risque':10s}")
print("-"*70)

infos = [
    ("Scène 1 — Route calme",       scene_1, outils_1, rapport_1),
    ("Scène 2 — Piéton proche",     scene_2, outils_2, rapport_2),
    ("Scène 3 — Intersection",      scene_3, outils_3, rapport_3),
]

for nom, scene, outils, rapport in infos:
    risque = extraire_risque(rapport)
    print(f"{nom:25s} {len(scene):8d} {len(set(outils)):8d} {risque:10s}")

print("="*70)
print("\n✅ L'agent appelle bien ses 3 outils à chaque analyse")
print("✅ Le niveau de risque s'adapte au contexte de la scène")
print("✅ Les recommandations sont spécifiques aux conditions nocturnes")


- Notre agent appelle systématiquement ses 3 outils avant de répondre
- Le niveau de risque est calibré selon les règles du code de la route
- Les recommandations sont adaptées aux conditions nocturnes
- Le format de rapport est reproductible


lancer l'interface Streamlit pour la démo interactive

```
streamlit run src/interface/app.py
```
